# SBND RWM Timing Calibration (Run 1 + Run 2)

## 1. Setup

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../lib')
import paths
from calib_fit_helpers import *
from run_table import build_run_table, consecutive_groups, build_fine_base
import event_timestamps_run1 as ev1
import event_timestamps_run2 as ev2

# ===== Set these paths =====
NTUPLE = {1: '/path/to/run1_timing_calib_ntuple.root',   # input ntuple per run (README: Data on Fermilab)
          2: '/path/to/run2_timing_calib_ntuple.root'}
CALIB_DIR = '../output'   # calibration notebook output folder: CALIB_DIR/run{1,2}/sbnd_fine_calibration.csv
paths.set_paths(NTUPLE, CALIB_DIR)

RUNS = (1, 2)

CHUNK_TARGET_ENTRIES = 18000
MIN_CHUNK_ENTRIES = 8000
RECOVERY_MIN_ENTRIES = 2000
SPOTTY_BEAM_MIN_GAP = 45

CFG = {
    1: dict(ev=ev1, out=paths.RUN1_SBND),
    2: dict(ev=ev2, out=paths.RUN2_SBND),
}
FIT_KW = dict(weighted_period=PERIOD_WEIGHTED, poisson_weight=POISSON_WEIGHT, sigma_fit_window=SIGMA_FIT_WINDOW,
              recovery_min_entries=RECOVERY_MIN_ENTRIES)

### Where the Input Ntuples Come From

| | |
|---|---|
| Analysis module | [`TimingCalibAna`](https://github.com/SBNSoftware/sbndcode/tree/lnguyen/timing_calibration_module/sbndcode/TimingCalib) (`sbndcode/TimingCalib`, fcl `timingcalibana.fcl`), sbndcode feature branch [`lnguyen/timing_calibration_module`](https://github.com/SBNSoftware/sbndcode/tree/lnguyen/timing_calibration_module) |
| Release | Gen2 production `v10_14_02_10`, sbndcode branch [`production/sbnd-gen2`](https://github.com/SBNSoftware/sbndcode/tree/production/sbnd-gen2) |
| Run 1 input (SAM) | `data_MCP2025C_Fall25-Run1_BNB_RollingDev_bnblight_v10_14_02_reco1_sbnd`, `data_MCP2025C_Fall25-Run1_BNB_FixedDev_bnblight_v10_14_02_reco1_sbnd`, `data_SBND2026A_gen2_BNB-Run1_v10_14_02_02_reco1_sbnd` |
| Run 2 input (SAM) | `data_SBND2026A_gen2_InTime-Run2b_beamlight_v10_14_02_06_reco1_sbnd`, `data_SBND2026A_gen2_InTime-Run2c_beamlight_v10_14_02_06_reco1_sbnd` |
| Output tree | `timingcalib/timingCalibTree`: one row per ν-candidate slice, uncalibrated times |
| Input ntuples | `/pnfs/sbnd/persistent/users/lnguyen/RWM_Calibration/run{1,2}_timing_calib_ntuple.root` |
| Output calibration CSVs | `/pnfs/sbnd/persistent/users/lnguyen/RWM_Calibration/run{1,2}_calibration.csv` (this notebook's output from these ntuples; tech note results) |
| Known gap | the production had failed jobs: about 2 % of slices missing |

## 2. Load Data

In [ ]:
S = {}
for run in RUNS:
    c = CFG[run]
    df = load_run_raw(run)
    dfrun = build_run_table(df)
    S[run] = dict(df=df, dfrun=dfrun)
    print(f"Run {run}: {len(df)} slices, {len(dfrun)} runs; registry: bad={len(c['ev'].BAD_RUNS)}, splits={len(c['ev'].FINE_SPLITS)}")

## 3. Run Groups and Splits

- Registry splits + bad runs: `FINE_SPLITS`, `BAD_RUNS` in `lib/event_timestamps_run{1,2}.py`
- Added here: split at a gap in run numbers, and at a mid-run beam gap ≥ `SPOTTY_BEAM_MIN_GAP` min

In [ ]:
for run in RUNS:
    s, ev = S[run], CFG[run]['ev']
    good = sorted(r for r in s['dfrun']['run'].values if r not in ev.BAD_RUNS)
    groups = consecutive_groups(good)
    s['fine_base'], n_spotty, n_gaps = build_fine_base(s['df'], s['dfrun'], groups, ev.FINE_SPLITS, SPOTTY_BEAM_MIN_GAP, counts=True)
    n_multi = sum(1 for g in groups if len(g) > 1)
    print(f'Run {run}: {len(good)} good runs (excluding {len(ev.BAD_RUNS)} bad) -> {len(groups)} consecutive-run groups '
          f'({n_multi} multi-run) -> {len(s["fine_base"])} rows after splits ({n_spotty} spotty-beam splits >= {SPOTTY_BEAM_MIN_GAP} min of {n_gaps} found)')

## 4. Chunk and Fit

In [ ]:
for run in RUNS:
    s, c = S[run], CFG[run]
    s['fine_df'], skipped, s['reasons'] = build_fine_chunks(s['fine_base'], s['df'], CHUNK_TARGET_ENTRIES, MIN_CHUNK_ENTRIES, **FIT_KW)
    f = s['fine_df']
    print(f'Run {run}: {len(f)} chunks from {len(s["fine_base"])} rows, skipped {skipped}:')
    for start, end, entries, reason, _ in s['reasons']:
        print(f'  {start:%d-%b-%y %H:%M} to {end:%d-%b-%y %H:%M} ({entries} entries): {reason}')
    print(f'  entries per chunk: min={f["entries"].min()}, median={f["entries"].median():.0f}, max={f["entries"].max()}')

## 5. Export to CSV

In [ ]:
for run in RUNS:
    s, c = S[run], CFG[run]
    prod = production_table(s['fine_df'], s['reasons'], PRODUCTION_COLS, labels=c['ev'].chunk_labels)
    paths.write_csv(prod, f'{c["out"]}/sbnd_fine_calibration.csv')
    print(f'Run {run}: wrote sbnd_fine_calibration.csv ({len(prod)} rows: {prod["constants"].value_counts().to_dict()}, '
          f'bunch_condition {prod["bunch_condition"].value_counts().to_dict()}, segment {prod["segment"].value_counts().sort_index().to_dict()})')